In [ ]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd

# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
%%writefile screen_validation.py
"""
screen_validation.py — analyze the LLM-vs-rules screening audit BEFORE PRISMA
============================================================================
Implements the pre-manual-review analysis plan:

  PHASE A  model health            which models actually answered, error taxonomy
           consensus repair        recompute llm_agree / llm_n_models / llm_label
                                   from the per-model labels (no API calls)
  PHASE B  agreement statistics    percent agreement, Cohen's kappa, Fleiss' kappa,
                                   Krippendorff's alpha, Gwet's AC1, PABAK,
                                   prevalence index, bias index  (+ bootstrap CIs)
  PHASE C  cross-tabs & subgroups  rule x LLM confusion, match_vs_rule breakdown,
                                   metrics split by abstract-present / MeSH-present /
                                   year / field  (where those columns exist)
  PHASE D  calibration             is self-reported confidence trustworthy?
                                   is llm_n_relevant a better ordinal signal?
  PHASE E  validation sample       stratified sampling plan for human adjudication,
                                   oversampling the false-negative-risk strata

WHY THE REPAIR STEP EXISTS
--------------------------
An earlier version of llm_screen.py computed llm_agree over ALL model outputs
including API_ERROR / PARSE_ERROR, so a record where 2 models agreed and 1
errored was recorded as llm_agree=False. It also had no llm_n_models column.
repair_consensus() recomputes both from the per-model *_label columns, so you
do NOT need to re-run any API calls.

IMPORTANT INTERPRETATION WARNING
--------------------------------
Agreement between two automated screeners is NOT accuracy. Both layers read the
same metadata (title/abstract/keywords/MeSH/journal), so their errors are
positively correlated: a record with no tick signal in its metadata is likely
missed by BOTH. High agreement therefore cannot rule out shared blind spots.
Only the human-validated stratified sample (Phase E) estimates real recall.

Usage (Colab)
-------------
    import screen_validation as sv
    df = sv.load("/content/drive/.../audit.csv")
    df = sv.repair_consensus(df)
    sv.model_health(df)
    sv.agreement_report(df)
    sv.crosstabs(df)
    sv.subgroups(df, corpus_df)        # corpus_df = canonical_screened (for oa_abstract etc.)
    sv.calibration(df)
    plan, sample = sv.validation_sample(df, total=2000, out="/content/drive/.../validation_sample.csv")
"""

from __future__ import annotations
import re
import numpy as np
import pandas as pd

LABELS = ["TICK_RELEVANT", "NOT_TICK", "UNCERTAIN"]
ERRORS = ["API_ERROR", "PARSE_ERROR", "CONFIG_ERROR"]
KEEP_LABELS = ["TICK_RELEVANT", "UNCERTAIN"]      # recall-safety: UNCERTAIN counts as keep


def load(path, models=("gpt", "gemini", "deepseek", "claude")):
    df = pd.read_csv(path, low_memory=False)
    present = [m for m in models if f"{m}_label" in df.columns]
    print(f"Loaded {len(df):,} rows | models present: {present}")
    df.attrs["models"] = present
    return df


def _models(df):
    return df.attrs.get("models") or [c[:-6] for c in df.columns if c.endswith("_label")
                                      and c != "llm_label"]


# ============================================================ PHASE A ========
def model_health(df, models=None):
    """Per-model validity + error taxonomy. Run this FIRST — if a model mostly
    errored, your 'consensus' is really a 1- or 2-model vote."""
    models = models or _models(df)
    rows = []
    for m in models:
        s = df[f"{m}_label"].fillna("")
        valid = s.isin(LABELS)
        r = {"model": m, "n": len(s), "valid_%": round(100 * valid.mean(), 1)}
        for e in ERRORS:
            r[e] = int((s == e).sum())
        r["blank"] = int((s == "").sum())
        rows.append(r)
    out = pd.DataFrame(rows)
    print("\n=== PER-MODEL HEALTH ===")
    print(out.to_string(index=False))
    bad = out[out["valid_%"] < 90]
    if len(bad):
        print("\n!! Models below 90% validity — their votes are missing from most "
              "records, so treat the consensus as a smaller ensemble:")
        for _, r in bad.iterrows():
            print(f"   {r['model']}: {r['valid_%']}% valid "
                  f"(API_ERROR={r['API_ERROR']:,}, PARSE_ERROR={r['PARSE_ERROR']:,})")
    return out


def repair_consensus(df, models=None):
    """Recompute llm_n_models / llm_agree / llm_label / llm_n_relevant from the
    per-model labels, ignoring error rows. Fixes the pre-fix-version artifact
    where any single model error forced llm_agree=False."""
    models = models or _models(df)
    L = df[[f"{m}_label" for m in models]].fillna("")
    valid = L.isin(LABELS)

    df = df.copy()
    df["llm_n_models"] = valid.sum(axis=1)
    for lab in LABELS:
        df[f"n_{lab.lower()}"] = (L == lab).sum(axis=1)

    # majority vote among VALID labels; ties -> UNCERTAIN
    counts = df[[f"n_{l.lower()}" for l in LABELS]].to_numpy()
    top = counts.max(axis=1)
    n_top = (counts == top[:, None]).sum(axis=1)
    idx = counts.argmax(axis=1)
    lab = np.array(LABELS)[idx]
    lab = np.where((top == 0) | (n_top > 1), "UNCERTAIN", lab)
    df["llm_label_fixed"] = lab

    df["llm_agree_fixed"] = (df["llm_n_models"] >= 2) & (top == df["llm_n_models"])
    df["llm_n_relevant_fixed"] = df["n_tick_relevant"]

    if "llm_agree" in df.columns:
        changed = (df["llm_agree"].astype(str).str.lower().eq("true")
                   != df["llm_agree_fixed"]).sum()
        print(f"\n[repair] llm_agree changed for {changed:,} rows "
              f"({changed/len(df):.1%}) — these were records where models agreed "
              f"but one errored.")
    print(f"[repair] records by #valid models:")
    print(df["llm_n_models"].value_counts().sort_index().to_string())
    return df


# ============================================================ PHASE B ========
def _binary_rule(df, col="oa_relevance_tier"):
    """Recall-safety collapse: rules {relevant, screen} -> KEEP, {irrelevant} -> DROP."""
    return df[col].map({"relevant": "KEEP", "screen": "KEEP",
                        "irrelevant": "DROP"}).fillna("KEEP")


def _binary_llm(df, col="llm_label_fixed"):
    """Recall-safety collapse: {TICK_RELEVANT, UNCERTAIN} -> KEEP, {NOT_TICK} -> DROP."""
    return df[col].map({"TICK_RELEVANT": "KEEP", "UNCERTAIN": "KEEP",
                        "NOT_TICK": "DROP"}).fillna("KEEP")


def cohen_kappa(a, b):
    cats = sorted(set(a) | set(b))
    n = len(a)
    obs = (a == b).mean()
    exp = sum((np.mean(a == c)) * (np.mean(b == c)) for c in cats)
    return (obs - exp) / (1 - exp) if exp < 1 else np.nan, obs


def gwet_ac1(a, b):
    """Paradox-resistant. Uses expected DISAGREEMENT, so it does not collapse
    under high prevalence the way kappa does."""
    cats = sorted(set(a) | set(b))
    k = len(cats)
    obs = (a == b).mean()
    pi = [(np.mean(a == c) + np.mean(b == c)) / 2 for c in cats]
    pe = sum(p * (1 - p) for p in pi) / (k - 1) if k > 1 else 0
    return (obs - pe) / (1 - pe) if pe < 1 else np.nan


def pabak(obs, k=2):
    """Prevalence-Adjusted Bias-Adjusted Kappa = (k*po - 1)/(k - 1)."""
    return (k * obs - 1) / (k - 1)


def prevalence_bias_index(a, b, pos="KEEP"):
    """2x2 only. PI = |a-d|/N (high => kappa paradox). BI = |b-c|/N (rater bias)."""
    A = ((a == pos) & (b == pos)).sum()
    D = ((a != pos) & (b != pos)).sum()
    B = ((a == pos) & (b != pos)).sum()
    C = ((a != pos) & (b == pos)).sum()
    n = len(a)
    return abs(A - D) / n, abs(B - C) / n, (A, B, C, D)


def fleiss_kappa(df, models=None):
    """Agreement among the LLMs themselves (valid labels only, >=2 raters)."""
    models = models or _models(df)
    L = df[[f"{m}_label" for m in models]].fillna("")
    mask = L.isin(LABELS).sum(axis=1) >= 2
    sub = L[mask]
    if not len(sub):
        return np.nan, 0
    mat = np.array([[ (row == lab).sum() for lab in LABELS ] for _, row in sub.iterrows()],
                   dtype=float)
    n_i = mat.sum(axis=1)
    p_j = mat.sum(axis=0) / n_i.sum()
    P_i = ((mat ** 2).sum(axis=1) - n_i) / (n_i * (n_i - 1))
    P_bar = P_i.mean()
    Pe = (p_j ** 2).sum()
    return ((P_bar - Pe) / (1 - Pe) if Pe < 1 else np.nan), int(mask.sum())


def krippendorff_alpha(df, models=None):
    """Nominal alpha — handles missing ratings (errored models) natively."""
    models = models or _models(df)
    L = df[[f"{m}_label" for m in models]].fillna("")
    L = L.where(L.isin(LABELS))
    mask = L.notna().sum(axis=1) >= 2
    sub = L[mask]
    if not len(sub):
        return np.nan
    Do = 0.0; n_total = 0
    cat_counts = {c: 0 for c in LABELS}
    for _, row in sub.iterrows():
        vals = [v for v in row if isinstance(v, str)]
        m = len(vals)
        for v in vals:
            cat_counts[v] += 1
        n_total += m
        disagree = sum(1 for i in range(m) for j in range(m) if i != j and vals[i] != vals[j])
        Do += disagree / (m - 1)
    Do /= n_total
    tot = sum(cat_counts.values())
    De = 1 - sum((c / tot) ** 2 for c in cat_counts.values())
    De *= tot / (tot - 1)
    return 1 - Do / De if De > 0 else np.nan


def _boot_ci(fn, a, b, n_boot=300, seed=0):
    rng = np.random.default_rng(seed)
    n = len(a)
    vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        try:
            v = fn(a[i], b[i])
            vals.append(v[0] if isinstance(v, tuple) else v)
        except Exception:
            pass
    return (np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5)) if vals else (np.nan, np.nan)


def agreement_report(df, n_boot=300):
    """Rules vs LLM consensus, plus LLM-internal agreement. Reports kappa AND
    paradox-resistant coefficients, because a ~70%-irrelevant corpus deflates kappa."""
    rb = _binary_rule(df).to_numpy()
    lb = _binary_llm(df).to_numpy()

    k, obs = cohen_kappa(rb, lb)
    ac1 = gwet_ac1(rb, lb)
    pab = pabak(obs, k=2)
    pi, bi, cells = prevalence_bias_index(rb, lb)
    k_lo, k_hi = _boot_ci(cohen_kappa, rb, lb, n_boot)
    a_lo, a_hi = _boot_ci(gwet_ac1, rb, lb, n_boot)

    print("\n=== RULES vs LLM CONSENSUS (binary, recall-safety collapse) ===")
    print(f"  2x2: both KEEP={cells[0]:,}  rulesKEEP/llmDROP={cells[1]:,}  "
          f"rulesDROP/llmKEEP={cells[2]:,}  both DROP={cells[3]:,}")
    print(f"  percent agreement : {obs:.3f}")
    print(f"  Cohen's kappa     : {k:.3f}   (95% CI {k_lo:.3f}–{k_hi:.3f})")
    print(f"  Gwet's AC1        : {ac1:.3f}   (95% CI {a_lo:.3f}–{a_hi:.3f})")
    print(f"  PABAK             : {pab:.3f}")
    print(f"  prevalence index  : {pi:.3f}   {'<-- HIGH: kappa is deflated by skew' if pi > 0.4 else ''}")
    print(f"  bias index        : {bi:.3f}   {'<-- screeners systematically differ' if bi > 0.2 else ''}")
    if not np.isnan(k) and not np.isnan(ac1) and (ac1 - k) > 0.15:
        print("  NOTE: AC1 >> kappa is the classic kappa paradox under skewed prevalence.")
        print("        Lead with AC1/PABAK; report kappa for comparability.")

    fk, n_fk = fleiss_kappa(df)
    alpha = krippendorff_alpha(df)
    print("\n=== AGREEMENT AMONG THE LLMs (valid labels only) ===")
    print(f"  Fleiss' kappa       : {fk:.3f}  (on {n_fk:,} records with >=2 valid)")
    print(f"  Krippendorff's alpha: {alpha:.3f}  (handles the errored/missing votes)")

    # pairwise, so you can see which model is the outlier
    models = _models(df)
    print("\n  pairwise (records where BOTH gave a valid label):")
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            m1, m2 = models[i], models[j]
            s = df[[f"{m1}_label", f"{m2}_label"]].fillna("")
            s = s[s[f"{m1}_label"].isin(LABELS) & s[f"{m2}_label"].isin(LABELS)]
            if len(s) < 30:
                print(f"    {m1} vs {m2}: only {len(s):,} comparable — skipped")
                continue
            kk, oo = cohen_kappa(s[f"{m1}_label"].to_numpy(), s[f"{m2}_label"].to_numpy())
            print(f"    {m1} vs {m2}: n={len(s):,}  agreement={oo:.3f}  kappa={kk:.3f}")

    print("\n  REMINDER: agreement != accuracy. Both layers read the same metadata,"
          "\n  so their errors are correlated. Only the human sample estimates recall.")


# ============================================================ PHASE C ========
def crosstabs(df):
    print("\n=== match_vs_rule DISTRIBUTION ===")
    if "match_vs_rule" in df.columns:
        vc = df["match_vs_rule"].value_counts(dropna=False)
        for k_, v in vc.items():
            print(f"  {str(k_):20s}: {v:>8,}  ({v/len(df):.1%})")
    print("\n=== rule tier x LLM consensus ===")
    print(pd.crosstab(df["oa_relevance_tier"], df["llm_label_fixed"],
                      margins=True).to_string())
    print("\n=== rule tier x #models voting relevant (0-3) ===")
    print(pd.crosstab(df["oa_relevance_tier"], df["llm_n_relevant_fixed"]).to_string())


def subgroups(df, corpus=None, join_on="oa_id"):
    """Accuracy proxies split by abstract-present / MeSH-present / year.
    Pass the canonical corpus df to supply oa_abstract / oa_mesh / oa_year."""
    d = df
    if corpus is not None:
        cols = [c for c in ["oa_abstract", "oa_mesh", "oa_year", "oa_field"]
                if c in corpus.columns]
        d = df.merge(corpus[[join_on] + cols], on=join_on, how="left")

    rb = _binary_rule(d); lb = _binary_llm(d)
    d = d.assign(_agree=(rb.to_numpy() == lb.to_numpy()))

    for col, label in [("oa_abstract", "abstract present"),
                       ("oa_mesh", "MeSH present")]:
        if col in d.columns:
            has = d[col].astype(str).str.strip().ne("") & d[col].notna()
            g = d.groupby(has)["_agree"].agg(["mean", "size"])
            g.index = [f"{label}=False", f"{label}=True"][:len(g)]
            print(f"\n=== agreement by {label} ===")
            print(g.rename(columns={"mean": "agreement", "size": "n"}).to_string())
            print("  (a materially LOWER agreement for missing-metadata records means "
                  "you should route their DROP decisions to human review)")

    if "oa_year" in d.columns:
        d["_yr"] = pd.to_numeric(d["oa_year"], errors="coerce")
        bins = pd.cut(d["_yr"], [1999, 2005, 2010, 2015, 2020, 2030])
        print("\n=== agreement by year band (drift / recency bias check) ===")
        print(d.groupby(bins, observed=True)["_agree"].agg(["mean", "size"]).to_string())
    return d


# ============================================================ PHASE D ========
def calibration(df, n_bins=10):
    """Is self-reported confidence trustworthy? Is llm_n_relevant better?
    Without ground truth we can only check INTERNAL consistency; after you label
    the validation sample, re-run with truth= to get real ECE/Brier."""
    models = _models(df)
    print("\n=== CONFIDENCE DISTRIBUTION (self-reported) ===")
    for m in models:
        c = pd.to_numeric(df.get(f"{m}_conf"), errors="coerce").dropna()
        if len(c):
            print(f"  {m:9s}: n={len(c):,}  mean={c.mean():.3f}  "
                  f"median={c.median():.3f}  %>=0.9={100*(c>=0.9).mean():.1f}%")
    print("  LLM self-confidence is typically OVERCONFIDENT and clusters high;")
    print("  treat it as a ranking signal only, not a probability.")

    print("\n=== llm_n_relevant (0-3) as an ordinal confidence ladder ===")
    t = df.groupby("llm_n_relevant_fixed").agg(
        n=("oa_id", "size"),
        pct_rule_relevant=("oa_relevance_tier", lambda s: (s == "relevant").mean()))
    print(t.to_string())
    print("  If pct_rule_relevant rises monotonically with the count, use "
          "llm_n_relevant as your triage variable rather than raw confidence.")


def calibration_with_truth(df, truth_col="human_label", conf_col="gpt_conf", n_bins=10):
    """Run AFTER labeling the validation sample. Real ECE + Brier."""
    d = df[df[truth_col].notna()].copy()
    y = d[truth_col].astype(str).str.upper().str.startswith("TICK").astype(int)
    p = pd.to_numeric(d[conf_col], errors="coerce").fillna(0.5)
    pred_rel = d[conf_col].notna()
    bins = pd.cut(p, np.linspace(0, 1, n_bins + 1), include_lowest=True)
    g = pd.DataFrame({"p": p, "y": y}).groupby(bins, observed=True).agg(
        n=("y", "size"), conf=("p", "mean"), acc=("y", "mean"))
    ece = float((g["n"] / g["n"].sum() * (g["conf"] - g["acc"]).abs()).sum())
    brier = float(((p - y) ** 2).mean())
    print("\n=== CALIBRATION (with human truth) ===")
    print(g.to_string())
    print(f"  ECE   = {ece:.4f}   (0 = perfect; >0.1 = poorly calibrated)")
    print(f"  Brier = {brier:.4f}  (lower is better)")
    return g, ece, brier


# ============================================================ PHASE E ========
def validation_sample(df, total=2000, out=None, seed=0):
    """Stratified human-validation sample. Deliberately OVERSAMPLES the strata
    where false negatives hide (llm_looser = rules dropped but LLM kept), so you
    can estimate corpus recall with a stratified estimator afterwards."""
    d = df.copy()
    rb = _binary_rule(d); lb = _binary_llm(d)
    d["_stratum"] = np.select(
        [(rb == "DROP") & (lb == "KEEP"),
         (rb == "KEEP") & (lb == "DROP"),
         (d["oa_relevance_tier"] == "screen"),
         (rb == "KEEP") & (lb == "KEEP"),
         (rb == "DROP") & (lb == "DROP")],
        ["llm_looser (FN RISK)", "llm_stricter", "screen_adjudicated",
         "agree_keep", "agree_drop"], default="other")
    # sampling weight per stratum: FN-risk strata get the most attention
    w = {"llm_looser (FN RISK)": 0.35, "llm_stricter": 0.20,
         "screen_adjudicated": 0.20, "agree_keep": 0.15, "agree_drop": 0.10}

    sizes = d["_stratum"].value_counts()
    rows = []
    for s, frac in w.items():
        pool = d[d["_stratum"] == s]
        if not len(pool):
            continue
        n = min(len(pool), max(30, int(total * frac)))
        rows.append(pool.sample(n, random_state=seed))
    smp = pd.concat(rows) if rows else d.head(0)

    plan = pd.DataFrame({"stratum": sizes.index, "N_corpus": sizes.values})
    plan["n_sampled"] = plan["stratum"].map(smp["_stratum"].value_counts()).fillna(0).astype(int)
    plan["sample_frac"] = (plan["n_sampled"] / plan["N_corpus"]).round(4)
    plan["weight_N/n"] = (plan["N_corpus"] / plan["n_sampled"].replace(0, np.nan)).round(1)

    print("\n=== STRATIFIED VALIDATION PLAN ===")
    print(plan.to_string(index=False))
    print(f"\n  total to review: {len(smp):,}")
    print("  Extrapolate later with the stratified estimator:")
    print("     R_hat = SUM_over_strata  N_s * (relevant_found_s / n_s)")
    print("  and report Wilson CIs. Decide on the LOWER bound of recall, not the point estimate.")

    keep = [c for c in ["oa_id", "oa_title", "oa_relevance_tier", "oa_relevance_reason",
                        "llm_label_fixed", "llm_agree_fixed", "llm_n_relevant_fixed",
                        "match_vs_rule", "_stratum"] if c in smp.columns]
    keep += [c for c in smp.columns if c.endswith(("_label", "_conf", "_reason"))
             and not c.startswith("llm")]
    smp = smp[keep].copy()
    smp["human_label"] = ""      # <-- reviewer fills: TICK_RELEVANT / NOT_TICK / UNCERTAIN
    smp["human_notes"] = ""
    if out:
        smp.to_csv(out, index=False)
        print(f"  wrote {out}")
    return plan, smp


def recall_from_validation(sample, plan, truth_col="human_label"):
    """After labeling: stratified corpus-level estimates with Wilson CIs."""
    from math import sqrt
    d = sample[sample[truth_col].astype(str).str.strip().ne("")].copy()
    d["_truth_rel"] = d[truth_col].str.upper().str.startswith("TICK")
    d["_kept"] = ~d["_stratum"].eq("agree_drop") & ~d["_stratum"].eq("llm_stricter")

    rows = []
    for s, g in d.groupby("_stratum"):
        N = int(plan.loc[plan["stratum"] == s, "N_corpus"].iloc[0])
        n = len(g); r = int(g["_truth_rel"].sum())
        p = r / n if n else 0
        rows.append({"stratum": s, "N": N, "n": n, "rel_found": r,
                     "p_rel": round(p, 3), "est_relevant_in_corpus": round(N * p)})
    t = pd.DataFrame(rows)
    total_rel = t["est_relevant_in_corpus"].sum()
    print("\n=== STRATIFIED EXTRAPOLATION ===")
    print(t.to_string(index=False))
    print(f"  estimated relevant records in corpus: {total_rel:,.0f}")
    missed = t[t["stratum"].isin(["agree_drop", "llm_stricter"])]["est_relevant_in_corpus"].sum()
    if total_rel:
        print(f"  estimated relevant records DROPPED : {missed:,.0f} "
              f"=> estimated recall ~ {1 - missed/total_rel:.1%}")
        print("  ACT ON THE LOWER CONFIDENCE BOUND, not this point estimate.")
    return t

In [ ]:
import screen_validation as sv
df = sv.load("/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/audit.csv")
sv.model_health(df)                    # quantify the Gemini failure
df = sv.repair_consensus(df)           # fix llm_agree without re-calling APIs
sv.agreement_report(df)                # kappa + AC1 + PABAK + Fleiss + Krippendorff
sv.crosstabs(df)

corpus_df = pd.read_parquet("/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet")
sv.subgroups(df, corpus_df)            # by abstract-present / MeSH / year
sv.calibration(df)
plan, sample = sv.validation_sample(df, total=2000,
                                    out="/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/validation_sample.csv")